# Notebook 01: Data Cleaning Pipeline (Completed Solutions)

This notebook demonstrates the step-by-step cleaning of the messy dataset.
It executes each cleaning step, runs before/after checks, logs decisions, and exports the clean dataset to `data/clean/topic3-clean.csv`.

In [1]:
import sys
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.utils import load_raw_data, CLEAN_CSV_PATH

df_raw = load_raw_data(PROJECT_ROOT / "data" / "raw" / "topic3-data.csv")
print(f"Initial raw shape: {df_raw.shape}")

Initial raw shape: (480, 15)


## Step 1: Remove Exact Duplicate Rows
Rule: Keep the first occurrence of identical rows across all 15 columns. Dropped 16 rows.

In [2]:
df_dedup = df_raw.drop_duplicates(keep="first").copy()
print(f"Rows before: {len(df_raw)} -> Rows after: {len(df_dedup)} (Dropped: {len(df_raw) - len(df_dedup)})")
print(f"Distinct orders preserved: {df_dedup['order_id'].nunique()}")

Rows before: 480 -> Rows after: 464 (Dropped: 16)
Distinct orders preserved: 260


## Step 2: Date Standardization & Imputation
- Parse 4 mixed formats.
- Impute 6 bad dates (`2025-02-30`, `not recorded`) from sibling lines of the same order.
- Coerce `TBD` in `ship_date` to NaT.

In [3]:
df_dates = df_dedup.copy()
order_p = pd.to_datetime(df_dates["order_date"], format="mixed", errors="coerce")
ship_p = pd.to_datetime(df_dates["ship_date"], format="mixed", errors="coerce")

# Impute invalid order_dates from siblings in same order_id
bad_order = order_p.isna()
for idx in df_dates[bad_order].index:
    oid = df_dates.loc[idx, "order_id"]
    siblings = df_dates.loc[(df_dates["order_id"] == oid) & (~bad_order), "order_date"]
    if not siblings.empty:
        order_p.loc[idx] = pd.to_datetime(siblings.iloc[0], format="mixed", errors="coerce")

df_dates["order_date"] = order_p.dt.strftime("%Y-%m-%d")
df_dates["ship_date"] = ship_p.dt.strftime("%Y-%m-%d")
print(f"Valid order_dates: {df_dates['order_date'].notna().sum()} / {len(df_dates)}")
print(f"Valid ship_dates:  {df_dates['ship_date'].notna().sum()} (4 are TBD / awaiting shipment)")

Valid order_dates: 464 / 464
Valid ship_dates:  460 (4 are TBD / awaiting shipment)


## Step 3: Categorical Standardization & Region Imputation
- Clean city spellings (`SanDiego` -> `San Diego`).
- Impute and standardize region deterministically from city.
- Standardize subcategories and map category uniquely.
- Clean channels (`Web`, `E-Commerce` -> `Online`) and customer segments (`Corp` -> `Corporate`).

In [4]:
CITY_TO_REGION = {
    "Atlanta": "South", "Boston": "Northeast", "Chicago": "Midwest", "Dallas": "South",
    "Houston": "South", "Indianapolis": "Midwest", "Madison": "Midwest", "Milwaukee": "Midwest",
    "Nashville": "South", "New York": "Northeast", "Philadelphia": "Northeast", "Phoenix": "West",
    "Pittsburgh": "Northeast", "Portland": "West", "San Diego": "West", "Seattle": "West"
}
SUBCAT_TO_CAT = {
    "Chair": "Furniture", "Bookcase": "Furniture", "Desk": "Furniture",
    "Binder": "Office Supplies", "Paper": "Office Supplies", "Pen": "Office Supplies",
    "Headphones": "Technology", "Laptop": "Technology", "Monitor": "Technology"
}

df_cat = df_dates.copy()
df_cat["city"] = df_cat["city"].str.strip().replace({"SanDiego": "San Diego", "san diego": "San Diego"}).str.title()
df_cat["region"] = df_cat["city"].map(CITY_TO_REGION)
df_cat["subcategory"] = df_cat["subcategory"].str.strip().str.title()
df_cat["category"] = df_cat["subcategory"].map(SUBCAT_TO_CAT)

channel_map = {"web": "Online", "e-commerce": "Online", "online": "Online", "in store": "In-Store", "in-store": "In-Store", "instore": "In-Store", "phone": "Phone", "phone order": "Phone"}
df_cat["sales_channel"] = df_cat["sales_channel"].str.strip().str.lower().map(channel_map).fillna("Online")

cust_map = {"consumer": "Consumer", "corporate": "Corporate", "corp": "Corporate", "home office": "Home Office", "homeoffice": "Home Office"}
df_cat["customer_type"] = df_cat["customer_type"].str.strip().str.lower().map(cust_map).fillna("Consumer")

print("Distinct Regions:", sorted(df_cat["region"].unique()))
print("Distinct Categories:", sorted(df_cat["category"].unique()))
print("Distinct Channels:", sorted(df_cat["sales_channel"].unique()))
print("Distinct Customer Types:", sorted(df_cat["customer_type"].unique()))

Distinct Regions: ['Midwest', 'Northeast', 'South', 'West']
Distinct Categories: ['Furniture', 'Office Supplies', 'Technology']
Distinct Channels: ['In-Store', 'Online', 'Phone']
Distinct Customer Types: ['Consumer', 'Corporate', 'Home Office']


## Step 4: Discount & Return Flag Normalization

In [5]:
df_norm = df_cat.copy()

def parse_disc(val):
    if not val or pd.isna(val):
        return 0.0
    val = str(val).strip().lower()
    if val in ["", "none", "no discount", "0"]:
        return 0.0
    num = re.sub(r"[^\d.]", "", val)
    if "%" in val or "percent" in val:
        return float(num) / 100.0 if num else 0.0
    return float(val) if num else 0.0

df_norm["discount"] = df_norm["discount"].apply(parse_disc).round(2)

true_codes = {"1", "y", "yes", "true"}
df_norm["returned"] = df_norm["returned"].astype(str).str.strip().str.lower().isin(true_codes)
print("Discount tiers:", sorted(df_norm["discount"].unique()))
print("Returned counts:", df_norm["returned"].value_counts().to_dict())

Discount tiers: [np.float64(0.0), np.float64(0.05), np.float64(0.1), np.float64(0.15), np.float64(0.2)]
Returned counts: {False: 393, True: 71}


## Step 5: Clean Numeric Fields & Reconcile Revenue
- Parse quantities (convert number words, remove suffixes, impute missing).
- Clean unit prices (remove `$`, `USD`, commas, impute missing).
- Reconcile gross line revenue (`quantity * unit_price * (1 - discount)`) and net revenue.

In [6]:
df_clean = df_norm.copy()
WORD_TO_NUM = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}

# Quantity parsing
def clean_q(v):
    v = str(v).strip().lower()
    if v in WORD_TO_NUM:
        return float(WORD_TO_NUM[v])
    if v in [".", "", "unknown", "—", "none"]:
        return np.nan
    clean = re.sub(r"[^\d.]", "", v)
    try:
        return float(clean) if clean and clean != "." else np.nan
    except ValueError:
        return np.nan

df_clean["quantity"] = df_clean["quantity"].apply(clean_q)

# Unit price parsing
def clean_p(v):
    v = str(v).strip().lower()
    if v in [".", "", "unknown", "—", "none"]:
        return np.nan
    clean = re.sub(r"[^\d.]", "", v)
    try:
        return float(clean) if clean and clean != "." else np.nan
    except ValueError:
        return np.nan

df_clean["unit_price"] = df_clean["unit_price"].apply(clean_p)

# Recorded revenue parsing
def clean_r(v):
    s = str(v).strip()
    if s.lower() in ["unknown", "", "none", "."]:
        return np.nan
    is_neg = s.startswith("-") or ("(" in s and ")" in s)
    num = re.sub(r"[^\d.]", "", s)
    res = float(num) if num else np.nan
    return -res if (is_neg and res is not None) else res

df_clean["recorded_revenue"] = df_clean["revenue"].apply(clean_r)

# Impute missing quantity & unit_price via revenue formula
mq = df_clean["quantity"].isna() & df_clean["unit_price"].notna() & df_clean["recorded_revenue"].notna()
for idx in df_clean[mq].index:
    r = abs(df_clean.loc[idx, "recorded_revenue"])
    if df_clean.loc[idx, "recorded_revenue"] < 0:
        r = r / 0.60
    denom = df_clean.loc[idx, "unit_price"] * (1.0 - df_clean.loc[idx, "discount"])
    df_clean.loc[idx, "quantity"] = round(r / denom)

mp = df_clean["unit_price"].isna() & df_clean["quantity"].notna() & df_clean["recorded_revenue"].notna()
for idx in df_clean[mp].index:
    r = abs(df_clean.loc[idx, "recorded_revenue"])
    if df_clean.loc[idx, "recorded_revenue"] < 0:
        r = r / 0.60
    denom = df_clean.loc[idx, "quantity"] * (1.0 - df_clean.loc[idx, "discount"])
    df_clean.loc[idx, "unit_price"] = round(r / denom, 2)

df_clean["quantity"] = df_clean["quantity"].fillna(1).astype(int)
df_clean["unit_price"] = df_clean["unit_price"].fillna(df_clean.groupby("subcategory")["unit_price"].transform("median"))

# Formula reconciliation
df_clean["revenue"] = (df_clean["quantity"] * df_clean["unit_price"] * (1.0 - df_clean["discount"])).round(2)
df_clean["net_revenue"] = np.where(df_clean["returned"], 0.0, df_clean["revenue"])
df_clean["shipping_days"] = (pd.to_datetime(df_clean["ship_date"]) - pd.to_datetime(df_clean["order_date"])).dt.days

print(f"Total Gross Revenue: ${df_clean['revenue'].sum():,.2f}")
print(f"Total Net Revenue:   ${df_clean['net_revenue'].sum():,.2f}")

Total Gross Revenue: $179,532.37
Total Net Revenue:   $152,542.41


## Step 6: Save Cleaned Data and Final Inspection

In [7]:
cols = [
    "order_id", "order_date", "ship_date", "shipping_days",
    "region", "city", "category", "subcategory", "sales_channel",
    "customer_type", "quantity", "unit_price", "discount",
    "revenue", "net_revenue", "returned", "notes"
]
df_final = df_clean[cols]
df_final.to_csv(PROJECT_ROOT / "data" / "clean" / "topic3-clean.csv", index=False)
print(f"Successfully saved clean dataset: {df_final.shape}")
df_final.head(5)

Successfully saved clean dataset: (464, 17)


,order_id,order_date,ship_date,shipping_days,region,city,category,subcategory,sales_channel,customer_type,quantity,unit_price,discount,revenue,net_revenue,returned,notes
0,ORD-1001,2025-04-04,2025-04-11,7.0,Midwest,Chicago,Furniture,Chair,In-Store,Consumer,3,125.71,0.00,377.13,0.00,True,.
1,ORD-1002,2025-04-11,2025-04-14,3.0,West,San Diego,Office Supplies,Paper,Online,Home Office,1,14.04,0.15,11.93,11.93,False,priority
2,ORD-1002,2025-04-11,2025-04-14,3.0,West,San Diego,Technology,Laptop,Online,Home Office,1,902.88,0.05,857.74,857.74,False,
3,ORD-1002,2025-04-11,2025-04-14,3.0,West,San Diego,Office Supplies,Paper,Online,Home Office,1,11.36,0.00,11.36,11.36,False,gift
4,ORD-1003,2025-02-25,2025-03-01,4.0,South,Dallas,Office Supplies,Binder,Online,Home Office,2,6.92,0.10,12.46,0.00,True,
